# Evaluation

## 1. imports 

In [4]:
import pandas as pd

## 2. Load data

In [5]:
import json
import glob
from pathlib import Path
import pandas as pd

# ── Load the most recent JSON file from the articles folder ───────
ARTICLES_DIR = "../data/articles"

json_files = sorted(glob.glob(f"{ARTICLES_DIR}/*.json"), key=lambda p: Path(p).stat().st_mtime)
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {ARTICLES_DIR}")

json_path = json_files[-1]  # most recently created/modified file
print(f"Loading: {json_path}")

with open(json_path) as f:
    articles = json.load(f)

ALL_COMMODITIES = ["gold", "silver", "oil", "gas"]

# Flatten metadata + headline + body into rows
rows = []
for art in articles:
    meta = art["metadata"]
    row = {
        "obs_id":           meta["obs_id"],
        "article_date":     meta["article_date"],
        "commodity":        meta["commodity"],
        "n_words_target":   meta["n_words_target"],
        "references_break": meta.get("references_break"),
        "themes":           ", ".join(meta.get("themes", [])),
        "decoys_named":     ", ".join(meta.get("decoys_named", [])),
        "model":            meta["model"],
        "headline":         art["headline"],
        "body":             art["body"],
    }

    # one-hot commodity flags, read straight from metadata["commodities"]
    for c in ALL_COMMODITIES:
        row[c] = meta["commodities"].get(c, 0)

    # per-commodity price columns, only populated for commodities that
    # were actually sampled (others stay NaN)
    for c in ALL_COMMODITIES:
        entry = meta["prices"].get(c)
        row[f"current_price_{c}"] = entry["current_price"] if entry else None
        row[f"prices_21d_{c}"]    = entry["prices_21d"]    if entry else None

    rows.append(row)

df = pd.DataFrame(rows)

# reorder: metadata cols, then one-hot flags, then per-commodity prices, then text
meta_cols   = ["obs_id", "article_date", "commodity"]
onehot_cols = ALL_COMMODITIES
price_cols  = [f"{prefix}_{c}" for c in ALL_COMMODITIES
              for prefix in ("current_price", "prices_21d")]
other_cols  = ["references_break", "themes", "decoys_named", "n_words_target", "model"]
text_cols   = ["headline", "body"]

df = df[meta_cols + onehot_cols + price_cols + other_cols + text_cols]

out_path = str(Path(json_path).with_suffix(".csv"))
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print()

# ── Sanity check: distribution across commodity combinations ──────
combo_counts = df.apply(
    lambda r: "+".join(c for c in ALL_COMMODITIES if r[c] == 1), axis=1
).value_counts()
print("Commodity combination distribution:")
print(combo_counts)

n_commodities = df[ALL_COMMODITIES].sum(axis=1)
print(f"\nArticles by number of commodities: {n_commodities.value_counts().sort_index().to_dict()}")

Loading: ../data/articles/articles_all_commodities_baseline_seed7_20260722_190748.json
Saved: ../data/articles/articles_all_commodities_baseline_seed7_20260722_190748.csv
Shape: (5000, 22)

Commodity combination distribution:
gold                   1178
silver                  614
gold+silver             482
gas                     433
oil                     413
gold+oil                312
gold+gas                307
gold+silver+oil         208
silver+gas              190
gold+silver+gas         187
silver+oil              170
gold+silver+oil+gas     170
oil+gas                 137
gold+oil+gas            114
silver+oil+gas           85
Name: count, dtype: int64

Articles by number of commodities: {1: 2638, 2: 1598, 3: 594, 4: 170}


## 3. Eval XGBoost

### 3.1 template_gold_silver_struct_low_seed7_final

In [6]:
# import numpy as np
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.multioutput import MultiOutputClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report
# from xgboost import XGBClassifier

# ALL_COMMODITIES = ["gold", "silver", "oil", "gas"]

# # Combine headline + body as input text
# df["text"] = df["headline"].fillna("") + " " + df["body"].fillna("")

# # TF-IDF features
# vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
# X = vectorizer.fit_transform(df["text"])
# y = df[ALL_COMMODITIES].values

# # 70/10/20 train/val/test split: split off the 70% train first, then split
# # the remaining 30% into val (10% overall) / test (20% overall).
# X_train, X_rest, y_train, y_rest = train_test_split(
#     X, y, train_size=0.7, random_state=42)
# X_val, X_test, y_val, y_test = train_test_split(
#     X_rest, y_rest, train_size=1/3, random_state=42)

# print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

# # Multi-label XGBoost
# clf = MultiOutputClassifier(XGBClassifier(
#     n_estimators   = 200,
#     max_depth      = 6,
#     learning_rate  = 0.1,
#     eval_metric    = "logloss",
#     random_state   = 42,
#     tree_method    = "hist",
# ))
# clf.fit(X_train, y_train)

# for split_name, X_split, y_split in [("VAL", X_val, y_val), ("TEST", X_test, y_test)]:
#     print(f"\n{'='*20} {split_name} {'='*20}")
#     y_pred = clf.predict(X_split)

#     for i, c in enumerate(ALL_COMMODITIES):
#         print(f"=== {c.upper()} ===")
#         print(classification_report(y_split[:, i], y_pred[:, i]))

#     print(f"Exact match (all {len(ALL_COMMODITIES)} correct): {np.all(y_split == y_pred, axis=1).mean():.3f}")
#     print(f"Mean per-label accuracy: {(y_split == y_pred).mean():.3f}")

### 3.2 Hyperparameter optimization on the 500 val samples (Optuna)

In [7]:
import optuna

# Optimize XGBoost hyperparameters against the 500-sample val split from 3.1
# (X_train/y_train/X_val/y_val/X_test/y_test are already computed above).
# Objective: mean per-label accuracy on val, matching the metric printed in 3.1.

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 50, 400),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    model = MultiOutputClassifier(XGBClassifier(
        **params,
        eval_metric="logloss",
        random_state=42,
        tree_method="hist",
    ))
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    return (y_val == y_val_pred).mean()


study = optuna.create_study(direction="maximize", study_name="xgboost_commodity_multilabel")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"\nBest val mean per-label accuracy: {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-24 15:39:05,573] A new study created in memory with name: xgboost_commodity_multilabel
Best trial: 0. Best value: 0.976:   2%|▎         | 1/40 [00:47<31:06, 47.86s/it]

[I 2026-07-24 15:39:53,431] Trial 0 finished with value: 0.976 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.07429858939076715, 'subsample': 0.9368654411593162, 'colsample_bytree': 0.8438490894366867, 'min_child_weight': 9, 'reg_alpha': 0.7196858820164528, 'reg_lambda': 1.6484431063333487}. Best is trial 0 with value: 0.976.


Best trial: 1. Best value: 0.978:   5%|▌         | 2/40 [01:42<32:51, 51.89s/it]

[I 2026-07-24 15:40:48,134] Trial 1 finished with value: 0.978 and parameters: {'n_estimators': 97, 'max_depth': 9, 'learning_rate': 0.20745236539236106, 'subsample': 0.9025166554481062, 'colsample_bytree': 0.7244198509800188, 'min_child_weight': 7, 'reg_alpha': 0.15755956452647743, 'reg_lambda': 0.002335655926789636}. Best is trial 1 with value: 0.978.


Best trial: 2. Best value: 0.9795:   8%|▊         | 3/40 [05:10<1:15:48, 122.92s/it]

...

Best trial: 2. Best value: 0.9795:  28%|██▊       | 11/40 [24:38<1:08:14, 141.20s/it]

[I 2026-07-24 16:03:43,617] Trial 10 finished with value: 0.955 and parameters: {'n_estimators': 222, 'max_depth': 7, 'learning_rate': 0.010581492523709088, 'subsample': 0.6602996044539394, 'colsample_bytree': 0.6097905660246403, 'min_child_weight': 10, 'reg_alpha': 5.889521859821068, 'reg_lambda': 0.0758058856510741}. Best is trial 2 with value: 0.9795.


Best trial: 2. Best value: 0.9795:  28%|██▊       | 11/40 [28:56<1:16:17, 157.84s/it]


[W 2026-07-24 16:08:01,784] Trial 11 failed with parameters: {'n_estimators': 290, 'max_depth': 10, 'learning_rate': 0.07787734689606536, 'subsample': 0.8272032310497778, 'colsample_bytree': 0.9895709156042618, 'min_child_weight': 1, 'reg_alpha': 0.02014958783791254, 'reg_lambda': 0.0900239900598778} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_151657/684106220.py", line 25, in objective
    model.fit(X_train, y_train)
  File "/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/sklearn/multioutput.py", line 547, in fit
    super().fit(X, Y, sample_weight=sample_weight, **fit_params)
  File "/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/sklearn/base.py", line 

KeyboardInterrupt: 

### 3.3 Evaluation with the optimized parameters

In [ ]:
# Refit a final model on train with the tuned hyperparameters from 3.2, then
# evaluate on val (should match study.best_value) and on the held-out test split.
best_clf = MultiOutputClassifier(XGBClassifier(
    **study.best_params,
    eval_metric="logloss",
    random_state=42,
    tree_method="hist",
))
best_clf.fit(X_train, y_train)

for split_name, X_split, y_split in [("VAL", X_val, y_val), ("TEST", X_test, y_test)]:
    print(f"\n{'='*20} {split_name} (tuned) {'='*20}")
    y_pred = best_clf.predict(X_split)

    for i, c in enumerate(ALL_COMMODITIES):
        print(f"=== {c.upper()} ===")
        print(classification_report(y_split[:, i], y_pred[:, i]))

    print(f"Exact match (all {len(ALL_COMMODITIES)} correct): {np.all(y_split == y_pred, axis=1).mean():.3f}")
    print(f"Mean per-label accuracy: {(y_split == y_pred).mean():.3f}")